# Fix BiGG Synthetic Graph Masks

Patches already-generated BiGG synthetic graphs that have broken masks
(all-ones `train_masks`, all-zeros `val/test_masks`) by recomputing proper
train/val splits in-place, preserving the same proportions as the original graph.

Features, labels, and edges are **not touched** — only the mask tensors are replaced.

In [11]:
import torch
import dgl
import os
import glob

# Adjust PROJECT_ROOT if you run this notebook from a different location
PROJECT_ROOT  = os.path.abspath(os.path.join('..', '..'))
ORIGINAL_DIR  = os.path.join(PROJECT_ROOT, 'datasets', 'original')
SYNTHETIC_DIR = os.path.join(PROJECT_ROOT, 'datasets', 'synthetic', 'bigg')

print('Project root:', PROJECT_ROOT)
print('Original dir:', ORIGINAL_DIR)
print('Synthetic dir:', SYNTHETIC_DIR)

Project root: /home/jorgenwp/Koding/master/SynGraphBench
Original dir: /home/jorgenwp/Koding/master/SynGraphBench/datasets/original
Synthetic dir: /home/jorgenwp/Koding/master/SynGraphBench/datasets/synthetic/bigg


In [14]:
# List all synthetic BiGG graphs on disk
# Structure: SYNTHETIC_DIR/<dataset>/<task>/<file>
syn_files = sorted(glob.glob(os.path.join(SYNTHETIC_DIR, '*', '*', '*')))
print(f'Found {len(syn_files)} synthetic graph(s):\n')

current_dataset, current_task = None, None
for f in syn_files:
    rel = os.path.relpath(f, SYNTHETIC_DIR)
    dataset, task, fname = rel.split(os.sep)
    if dataset != current_dataset:
        print(f'{dataset}/')
        current_dataset = dataset
        current_task = None
    if task != current_task:
        print(f'  {task}/')
        current_task = task
    print(f'    {fname}')

Found 0 synthetic graph(s):



In [15]:
# -------------------------------------------------------------------
# Edit this dict: relative path from SYNTHETIC_DIR  →  original dataset name
# Format: '<dataset>/<task>/<file>'  →  '<original_dataset>'
# -------------------------------------------------------------------
TO_FIX = {
    'reddit/reddit_blksize_-1_b_1_lr_0.001_epochs_300': 'tolokers',
    # 'reddit/anomaly_detection/blksize_1024_b_1_lr_0.001_epochs_50': 'reddit',
    # add more entries as needed
}

In [16]:
def check_masks(path):
    """Print mask sums for a synthetic graph — run before and after fix."""
    g, _ = dgl.load_graphs(path)
    g = g[0]
    train_sum = g.ndata['train_masks'].sum().item()
    val_sum   = g.ndata['val_masks'].sum().item()
    test_sum  = g.ndata['test_masks'].sum().item()
    label = os.path.relpath(path, SYNTHETIC_DIR)
    print(f"{label:70s}  "
          f"train={int(train_sum):>8,}  val={int(val_sum):>8,}  test={int(test_sum):>8,}")

print('--- Before fix ---')
for syn_name in TO_FIX:
    check_masks(os.path.join(SYNTHETIC_DIR, syn_name))

--- Before fix ---
reddit/reddit_blksize_-1_b_1_lr_0.001_epochs_300                        train= 219,680  val=       0  test=       0


In [17]:
def fix_masks(syn_path, orig_name, seed=42):
    """
    Rewrite train/val/test masks on a synthetic DGL graph.

    Split proportions (n_train, n_val per column) are taken from the
    corresponding original graph. Nodes are assigned randomly with the
    given seed so results are reproducible. test_masks stay all-zeros.
    """
    torch.manual_seed(seed)

    syn_graphs, _ = dgl.load_graphs(syn_path)
    syn_g = syn_graphs[0]

    orig_graphs, _ = dgl.load_graphs(os.path.join(ORIGINAL_DIR, orig_name))
    orig_g = orig_graphs[0]

    num_nodes  = syn_g.num_nodes()
    num_splits = orig_g.ndata['train_masks'].shape[1]

    train_masks = torch.zeros(num_nodes, num_splits, dtype=torch.uint8)
    val_masks   = torch.zeros(num_nodes, num_splits, dtype=torch.uint8)
    test_masks  = torch.zeros(num_nodes, num_splits, dtype=torch.uint8)  # always zero

    for col in range(num_splits):
        n_train = int(orig_g.ndata['train_masks'][:, col].sum().item())
        n_val   = int(orig_g.ndata['val_masks'][:, col].sum().item())
        perm = torch.randperm(num_nodes)
        train_masks[perm[:n_train],              col] = 1
        val_masks  [perm[n_train:n_train+n_val], col] = 1

    syn_g.ndata['train_masks'] = train_masks
    syn_g.ndata['val_masks']   = val_masks
    syn_g.ndata['test_masks']  = test_masks

    dgl.save_graphs(syn_path, [syn_g])
    label = os.path.relpath(syn_path, SYNTHETIC_DIR)
    print(f"Fixed: {label}  "
          f"(train={train_masks.sum().item():,}, val={val_masks.sum().item():,})")

In [18]:
for syn_name, orig_name in TO_FIX.items():
    fix_masks(os.path.join(SYNTHETIC_DIR, syn_name), orig_name)

Fixed: reddit/reddit_blksize_-1_b_1_lr_0.001_epochs_300  (train=59,790, val=30,390)


In [19]:
print('--- After fix ---')
for syn_name in TO_FIX:
    check_masks(os.path.join(SYNTHETIC_DIR, syn_name))

--- After fix ---
reddit/reddit_blksize_-1_b_1_lr_0.001_epochs_300                        train=  59,790  val=  30,390  test=       0
